In [1]:
%%writefile app.py
import re
import gc
import torch
import faiss
import streamlit as st

from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


# -------------------- Streamlit configuration --------------------

st.set_page_config(
    page_title="YouTube AI Assistant",
    page_icon="🎬",
    layout="wide",
)

st.title("🎬 YouTube AI Assistant")
st.caption("Extract transcripts, summarize videos, and ask questions.")


# -------------------- Session state --------------------

if "messages" not in st.session_state:
    st.session_state.messages = []

if "video_data" not in st.session_state:
    st.session_state.video_data = None


# -------------------- YouTube URL extraction --------------------

def extract_video_id(url: str) -> str:
    url = url.strip()
    parsed = urlparse(url)

    host = (parsed.hostname or "").lower()

    if host.startswith("www."):
        host = host[4:]

    if host == "youtu.be":
        video_id = parsed.path.strip("/").split("/")[0]

    elif host in (
        "youtube.com",
        "m.youtube.com",
        "music.youtube.com",
    ):
        query = parse_qs(parsed.query)

        if query.get("v"):
            video_id = query["v"][0]
        else:
            parts = parsed.path.strip("/").split("/")

            if (
                len(parts) >= 2
                and parts[0] in ("shorts", "embed", "live", "v")
            ):
                video_id = parts[1]
            else:
                raise ValueError("Could not find a video ID.")

    else:
        raise ValueError("Please enter a valid YouTube URL.")

    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise ValueError("The YouTube video ID is invalid.")

    return video_id


# -------------------- Transcript extraction --------------------

def get_transcript(video_id: str, languages: list[str]) -> str:
    api = YouTubeTranscriptApi()
    fetched = api.fetch(
        video_id,
        languages=languages,
    )

    # Fixed: Access snippet.text directly since snippet is a dataclass object
    transcript = " ".join(
        snippet.text for snippet in fetched if hasattr(snippet, "text")
    )

    transcript = re.sub(r"\s+", " ", transcript).strip()

    if not transcript:
        raise ValueError("The transcript is empty.")

    return transcript


# -------------------- Text chunking --------------------

def chunk_text(
    text: str,
    chunk_size: int = 180,
    overlap: int = 30,
) -> list[str]:

    words = text.split()

    if not words:
        return []

    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Invalid chunk size or overlap.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])

        if chunk:
            chunks.append(chunk)

        if start + chunk_size >= len(words):
            break

    return chunks


# -------------------- Embedding model and FAISS --------------------

@st.cache_resource(show_spinner="Loading multilingual embedding model...")
def load_embedding_model():
    return SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )


def build_index(chunks, embedding_model):
    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    embeddings = embeddings.astype("float32")

    # Inner product on normalized embeddings = cosine similarity.
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index


def search_index(query, embedding_model, index, chunks, k=4):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    k = min(k, len(chunks))

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "text": chunks[int(idx)],
                "score": float(score),
                "chunk_id": int(idx) + 1,
            })

    return results


# -------------------- Language model --------------------

@st.cache_resource(show_spinner=False)
def load_llm(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if torch.cuda.is_available():
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        )

    else:
        # CPU inference can be very slow.
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
        )

    model.eval()

    return tokenizer, model


def generate_text(
    prompt,
    tokenizer,
    model,
    max_new_tokens=350,
):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful video assistant. "
                "Answer questions using only the supplied transcript "
                "context. If the answer is not present in the context, "
                "say that the transcript does not provide enough "
                "information. Do not invent facts. "
                "Respond in the same language as the user's question."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=6000,
    )

    # Important when the model is loaded on a GPU or multiple devices.
    input_device = model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Return only newly generated tokens, not the original prompt.
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()


# -------------------- RAG question answering --------------------

def ask(
    question,
    embedding_model,
    index,
    chunks,
    tokenizer,
    llm,
    k=4,
):
    question = question.strip()

    if not question:
        return "Please enter a question.", []

    top_chunks = search_index(
        question,
        embedding_model,
        index,
        chunks,
        k=k,
    )

    context = "\n\n".join(
        f"[Transcript chunk {item['chunk_id']}]\n{item['text']}"
        for item in top_chunks
    )

    prompt = f"""
Answer the user's question based on the transcript excerpts below.

If the transcript does not contain the answer, clearly say so.
Do not use outside knowledge to fill missing information.
Give a clear, concise answer.

TRANSCRIPT EXCERPTS:
{context}

USER QUESTION:
{question}
"""

    answer = generate_text(
        prompt,
        tokenizer,
        llm,
        max_new_tokens=350,
    )

    return answer, top_chunks


# -------------------- Sidebar settings --------------------

with st.sidebar:
    st.header("⚙️ Settings")

    language_choice = st.selectbox(
        "Transcript language",
        ["English", "Arabic", "English + Arabic"],
    )

    language_map = {
        "English": ["en"],
        "Arabic": ["ar"],
        "English + Arabic": ["en", "ar"],
    }

    model_choice = st.selectbox(
        "Language model",
        [
            "Qwen/Qwen2.5-3B-Instruct",
            "mistralai/Mistral-Nemo-Instruct-2407",
        ],
    )

    chunk_size = st.slider(
        "Chunk size (words)",
        min_value=80,
        max_value=400,
        value=180,
        step=20,
    )

    overlap = st.slider(
        "Chunk overlap (words)",
        min_value=10,
        max_value=80,
        value=30,
        step=5,
    )

    top_k = st.slider(
        "Retrieved chunks",
        min_value=1,
        max_value=6,
        value=4,
    )

    if overlap >= chunk_size:
        st.error("Overlap must be smaller than chunk size.")


# -------------------- Video loading UI --------------------

st.subheader("1. Load a YouTube video")

with st.form("video_form"):
    url = st.text_input(
        "YouTube URL",
        placeholder="https://www.youtube.com/watch?v=...",
    )

    load_video = st.form_submit_button(
        "Fetch transcript and build index",
        type="primary",
        use_container_width=True,
    )


if load_video:
    if overlap >= chunk_size:
        st.error("Overlap must be smaller than chunk size.")

    elif not url.strip():
        st.warning("Please enter a YouTube URL.")

    else:
        try:
            with st.spinner("Fetching the transcript..."):
                video_id = extract_video_id(url)
                transcript = get_transcript(
                    video_id,
                    language_map[language_choice],
                )

            with st.spinner("Preparing transcript chunks..."):
                chunks = chunk_text(
                    transcript,
                    chunk_size=chunk_size,
                    overlap=overlap,
                )

                if not chunks:
                    raise ValueError("No usable transcript chunks found.")

            embedding_model = load_embedding_model()

            with st.spinner("Building the semantic search index..."):
                index = build_index(chunks, embedding_model)

            st.session_state.video_data = {
                "video_id": video_id,
                "url": url.strip(),
                "transcript": transcript,
                "chunks": chunks,
                "index": index,
                "embedding_model": embedding_model,
                "model_name": model_choice,
                "chunk_size": chunk_size,
                "overlap": overlap,
            }

            st.session_state.messages = []

            st.success(
                f"Transcript loaded successfully! "
                f"{len(chunks)} chunks created."
            )

        except Exception as error:
            st.error(
                f"Could not load the video: {error}"
            )

            st.info(
                "Check that the URL is valid and the video has "
                "accessible captions in the selected language. "
                "Some videos restrict transcript access."
            )


# -------------------- Chat interface --------------------

data = st.session_state.video_data

if data:
    st.divider()

    st.subheader("2. Ask questions about the video")

    st.markdown(
        f"**Video ID:** `{data['video_id']}`  \n"
        f"**Transcript length:** {len(data['transcript'].split()):,} words  \n"
        f"**Chunks:** {len(data['chunks'])}"
    )

    with st.expander("View original transcript"):
        st.text_area(
            "Transcript",
            value=data["transcript"],
            height=250,
            disabled=True,
            label_visibility="collapsed",
        )

    # Clear history if the selected model changes.
    if data["model_name"] != model_choice:
        st.session_state.video_data["model_name"] = model_choice
        st.session_state.messages = []

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

            if message["role"] == "assistant" and message.get("sources"):
                with st.expander("View retrieved transcript chunks"):
                    for source in message["sources"]:
                        st.markdown(
                            f"**Chunk {source['chunk_id']} "
                            f"(similarity: {source['score']:.3f})**"
                        )
                        st.write(source["text"])

    question = st.chat_input(
        "Ask something about the video..."
    )

    if question:
        st.session_state.messages.append({
            "role": "user",
            "content": question,
        })

        with st.chat_message("user"):
            st.markdown(question)

        with st.chat_message("assistant"):
            try:
                with st.spinner("Loading the language model..."):
                    tokenizer, llm = load_llm(model_choice)

                with st.spinner("Finding relevant context and generating an answer..."):
                    answer, sources = ask(
                        question,
                        data["embedding_model"],
                        data["index"],
                        data["chunks"],
                        tokenizer,
                        llm,
                        k=top_k,
                    )

                st.markdown(answer)

                with st.expander("View retrieved transcript chunks"):
                    for source in sources:
                        st.markdown(
                            f"**Chunk {source['chunk_id']} "
                            f"(similarity: {source['score']:.3f})**"
                        )
                        st.write(source["text"])

                st.session_state.messages.append({
                    "role": "assistant",
                    "content": answer,
                    "sources": sources,
                })

            except Exception as error:
                st.error(f"Answer generation failed: {error}")

    if st.button("Clear chat history"):
        st.session_state.messages = []
        st.rerun()

else:
    st.info("Load a YouTube video above to start chatting with its transcript.")

Writing app.py


In [2]:
 !pip -q install \
    "transformers==4.52.4" \
    "youtube-transcript-api" \
    "sentence-transformers" \
    "faiss-cpu" \
    "streamlit" \
    "accelerate" \
    "bitsandbytes" \
    "pyngrok"

from pyngrok import ngrok
import subprocess
import time
import requests
import getpass

PORT = 8501

# 1. Enter your ngrok token securely
NGROK_TOKEN = getpass.getpass("Enter your new ngrok auth token: ")
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Close previous ngrok tunnels
ngrok.kill()

# 3. Stop the previous Streamlit process, if it exists
try:
    process.terminate()
    process.wait(timeout=5)
except (NameError, Exception):
    pass

# 4. Start Streamlit
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--browser.gatherUsageStats=false",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

# 5. Wait for Streamlit to become ready
local_url = f"http://127.0.0.1:{PORT}/_stcore/health"

for attempt in range(45):
    if process.poll() is not None:
        raise RuntimeError(
            "Streamlit stopped unexpectedly. "
            "Check that app.py exists and its dependencies are installed."
        )

    try:
        response = requests.get(local_url, timeout=2)

        if response.status_code == 200:
            print("Streamlit started successfully!")
            break

    except requests.RequestException:
        pass

    time.sleep(2)
else:
    process.terminate()
    raise TimeoutError(
        "Streamlit did not start within 90 seconds."
    )

# 6. Create the public ngrok URL
try:
    tunnel = ngrok.connect(PORT, "http")
    public_url = tunnel.public_url

    print("\nYour Streamlit app is ready!")
    print(public_url)

except Exception as error:
    process.terminate()
    raise RuntimeError(
        f"Could not create the ngrok tunnel: {error}"
    ) from error

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingfac